In [1]:
import json, os

with open("Phase4_Results/thresholds.json") as f:
    thresholds = json.load(f)

print("Categories in thresholds:", list(thresholds.keys()))
print("Sample threshold (Single-Concept):", thresholds["Single-Concept"])

Categories in thresholds: ['Ambiguous', 'Compositional', 'Conflicting', 'Negation', 'Single-Concept', '__global__']
Sample threshold (Single-Concept): {'n': 200, 'mu_var': 0.00235236, 'std_var': 0.00052036, 'T_var': 0.00339307, 'mu_clip': 0.258853, 'std_clip': 0.015338, 'T_clip': 0.228177}


In [ ]:
with open("Phase3_Data/prompt_config.json") as f:
    config = json.load(f)

print(f"Type: {type(config)}")

if isinstance(config, list):
    print(f"Total prompts: {len(config)}")
    print(f"First entry keys: {list(config[0].keys())}")
    print(f"First entry: {json.dumps(config[0], indent=2)}")
elif isinstance(config, dict):
    first_key = list(config.keys())[0]
    print(f"First key: {first_key}")
    print(f"First entry: {json.dumps(config[first_key], indent=2)}")

Type: <class 'list'>
Total prompts: 20
First entry keys: ['id', 'category', 'prompt', 'yolo_expected', 'yolo_counts', 'yolo_forbidden', 'manual', 'note']
First entry: {
  "id": "SC1",
  "category": "Single-Concept",
  "prompt": "A single red apple",
  "yolo_expected": [
    "apple"
  ],
  "yolo_counts": {
    "apple": 1
  },
  "yolo_forbidden": [],
  "manual": false,
  "note": "Exact count enforced. YOLO error if apple count != 1."
}


In [ ]:
import os
import json
import numpy as np

os.makedirs("Phase8_Data/prompts",      exist_ok=True)
os.makedirs("Phase8_Results",           exist_ok=True)

print("Phase 8 directories ready.")
print("Laptop notebook — Parts A, C, D only.")
print("Part B (generation) runs on Kaggle.")

Phase 8 directories ready.
Laptop notebook — Parts A, C, D only.
Part B (generation) runs on Kaggle.


In [ ]:
external_prompts = []


sc_prompts = [
    # Colour + object (YOLO-able)
    ("EX_SC01", "A red fire truck",          ["truck"],  {"truck":1}, False,
     "Color+object. YOLO: truck=1."),
    ("EX_SC02", "A yellow school bus",        ["bus"],    {"bus":1},   False,
     "Color+object. YOLO: bus=1."),
    ("EX_SC03", "A white cat",                ["cat"],    {"cat":1},   False,
     "Color+object. YOLO: cat=1."),
    ("EX_SC04", "A black dog",                ["dog"],    {"dog":1},   False,
     "Color+object. YOLO: dog=1."),
    ("EX_SC05", "A brown horse",              ["horse"],  {"horse":1}, False,
     "Color+object. YOLO: horse=1."),
    ("EX_SC06", "A green bicycle",            ["bicycle"],{"bicycle":1},False,
     "Color+object. YOLO: bicycle=1."),
    ("EX_SC07", "A red apple on a white surface",["apple"],{"apple":1},False,
     "Attribute binding with surface context."),
    ("EX_SC08", "A single yellow banana",     ["banana"], {"banana":1},False,
     "Exact count enforced. YOLO: banana=1."),
    ("EX_SC09", "A blue car",                 ["car"],    {"car":1},   False,
     "Color+object. YOLO: car=1."),
    ("EX_SC10", "A single orange",            ["orange"], {"orange":1},False,
     "Exact count. YOLO: orange=1."),
    # Texture + object (manual — texture not in COCO)
    ("EX_SC11", "A wooden chair",             [], {}, True,
     "Texture attribute. CLIP-only."),
    ("EX_SC12", "A metal spoon",              [], {}, True,
     "Material attribute. CLIP-only."),
    ("EX_SC13", "A glass bottle",             [], {}, True,
     "Material attribute. CLIP-only."),
    ("EX_SC14", "A fluffy white pillow",      [], {}, True,
     "Texture+colour. CLIP-only."),
    ("EX_SC15", "A shiny silver ring",        [], {}, True,
     "Material+colour. CLIP-only."),
    ("EX_SC16", "A rough stone wall",         [], {}, True,
     "Texture attribute. CLIP-only."),
    ("EX_SC17", "A smooth ceramic bowl",      [], {}, True,
     "Texture+material. CLIP-only."),
    ("EX_SC18", "A striped red and white sock",[], {}, True,
     "Pattern+colour attribute. CLIP-only."),
    ("EX_SC19", "A spotted yellow giraffe",   [], {}, True,
     "Pattern+colour. CLIP-only."),
    ("EX_SC20", "A transparent glass cup",    [], {}, True,
     "Material+transparency. CLIP-only."),
    # Shape + object
    ("EX_SC21", "A round red clock",          [], {}, True,
     "Shape+colour. CLIP-only."),
    ("EX_SC22", "A triangular yellow sign",   [], {}, True,
     "Shape+colour. CLIP-only."),
    ("EX_SC23", "A square blue picture frame",[], {}, True,
     "Shape+colour. CLIP-only."),
    ("EX_SC24", "A tall green cactus",        [], {}, True,
     "Size+colour. CLIP-only."),
    ("EX_SC25", "A small brown rabbit",       [], {}, True,
     "Size+colour. CLIP-only."),
    # Count-specific (tests exact counting)
    ("EX_SC26", "A single red rose",          [], {}, True,
     "Count constraint. CLIP-only (rose not in COCO)."),
    ("EX_SC27", "One white dove",             ["bird"], {"bird":1}, False,
     "Count constraint. YOLO: bird=1."),
    ("EX_SC28", "A single green leaf",        [], {}, True,
     "Count constraint. CLIP-only."),
    ("EX_SC29", "One lit candle",             [], {}, True,
     "Count+state. CLIP-only."),
    ("EX_SC30", "A single blue umbrella",     [], {}, True,
     "Count+colour. CLIP-only."),
    # Complex attributes
    ("EX_SC31", "A cracked brown egg",        [], {}, True,
     "State attribute. CLIP-only."),
    ("EX_SC32", "An open red book",           [], {}, True,
     "State+colour. CLIP-only."),
    ("EX_SC33", "A melting ice cream cone",   [], {}, True,
     "State attribute. CLIP-only."),
    ("EX_SC34", "A burning orange candle",    [], {}, True,
     "State+colour. CLIP-only."),
    ("EX_SC35", "A broken blue vase",         [], {}, True,
     "State+colour. CLIP-only."),
    ("EX_SC36", "A purple kite",              [], {}, True,
     "Colour attribute. CLIP-only."),
    ("EX_SC37", "A pink flamingo",            ["bird"], {"bird":1}, False,
     "Colour+species. YOLO: bird=1 (flamingo classified as bird)."),
    ("EX_SC38", "A golden trophy",            [], {}, True,
     "Material+colour. CLIP-only."),
    ("EX_SC39", "A red and white striped lighthouse", [], {}, True,
     "Pattern+colour. CLIP-only."),
    ("EX_SC40", "A single silver coin",       [], {}, True,
     "Count+material. CLIP-only."),
]

for (pid, prompt, ye, yc, manual, note) in sc_prompts:
    external_prompts.append({
        "id": pid, "category": "Single-Concept",
        "prompt": prompt,
        "yolo_expected": ye, "yolo_counts": yc,
        "yolo_forbidden": [], "manual": manual, "note": note
    })

# ════════════════════════════════════════════════════════
# CATEGORY 2: Compositional (40 prompts)
# Spatial relationships, object co-occurrence
# ════════════════════════════════════════════════════════
co_prompts = [
    # Spatial: on/above/below (YOLO where both objects in COCO)
    ("EX_CO01", "A dog sitting on a sofa",    ["dog","couch"], {"dog":1,"couch":1}, False,
     "Spatial: on. Both in COCO."),
    ("EX_CO02", "A cat sleeping on a bed",    ["cat","bed"],   {"cat":1,"bed":1},   False,
     "Spatial: on. Both in COCO."),
    ("EX_CO03", "A person standing next to a car", ["person","car"], {"person":1,"car":1}, False,
     "Spatial: next to. Both in COCO."),
    ("EX_CO04", "A bird perched on a bench",  ["bird"],        {"bird":1},          False,
     "Spatial: on. bench not in COCO."),
    ("EX_CO05", "A dog under a table",        ["dog","dining table"], {"dog":1,"dining table":1}, False,
     "Spatial: under. Both in COCO."),
    ("EX_CO06", "A cat inside a box",         ["cat"],         {"cat":1},           False,
     "Spatial: inside. box not in COCO."),
    ("EX_CO07", "An apple next to a banana",  ["apple","banana"], {"apple":1,"banana":1}, False,
     "Spatial: next to. Both in COCO."),
    ("EX_CO08", "A cup on a saucer",          ["cup"],         {"cup":1},           False,
     "Spatial: on. saucer not in COCO."),
    ("EX_CO09", "A backpack on a chair",      ["backpack","chair"], {"backpack":1,"chair":1}, False,
     "Spatial: on. Both in COCO."),
    ("EX_CO10", "A laptop on a desk",         ["laptop"],      {"laptop":1},        False,
     "Spatial: on. desk not in COCO."),
    # Spatial: manual (neither object in COCO)
    ("EX_CO11", "A flower in a vase",         [], {}, True,
     "Spatial: in. Neither in COCO. CLIP-only."),
    ("EX_CO12", "A tree beside a fence",      [], {}, True,
     "Spatial: beside. CLIP-only."),
    ("EX_CO13", "A moon above the clouds",    [], {}, True,
     "Spatial: above. CLIP-only."),
    ("EX_CO14", "A fish in a bowl",           [], {}, True,
     "Spatial: in. CLIP-only."),
    ("EX_CO15", "A kite above a field",       [], {}, True,
     "Spatial: above. CLIP-only."),
    # Attribute binding: two objects with colour
    ("EX_CO16", "A red ball and a blue cube", [], {}, True,
     "Two objects, colour attributes. CLIP-only."),
    ("EX_CO17", "A black cat and a white dog",["cat","dog"], {"cat":1,"dog":1}, False,
     "Two objects, colour attributes. Both in COCO."),
    ("EX_CO18", "A yellow taxi next to a red bus", ["car","bus"], {"bus":1}, False,
     "Two objects, colour attributes."),
    ("EX_CO19", "A green apple beside a red apple", ["apple"], {}, False,
     "Same object, two colours. YOLO: apple presence check."),
    ("EX_CO20", "A brown dog chasing a white cat", ["dog","cat"], {"dog":1,"cat":1}, False,
     "Action+two objects. Both in COCO."),
    # Non-spatial compositional
    ("EX_CO21", "A person reading a book",    ["person","book"], {"person":1,"book":1}, False,
     "Action+object. Both in COCO."),
    ("EX_CO22", "A child riding a bicycle",   ["person","bicycle"], {"person":1,"bicycle":1}, False,
     "Action+object. Both in COCO."),
    ("EX_CO23", "A man holding an umbrella",  ["person"],      {"person":1},        False,
     "Action+object. umbrella not in COCO."),
    ("EX_CO24", "A woman carrying a handbag", ["person"],      {"person":1},        False,
     "Action+object. handbag not in COCO."),
    ("EX_CO25", "A dog catching a frisbee",   ["dog","frisbee"], {"dog":1,"frisbee":1}, False,
     "Action+object. Both in COCO."),
    ("EX_CO26", "A cat watching a bird",      ["cat","bird"],  {"cat":1,"bird":1},  False,
     "Action+two objects. Both in COCO."),
    ("EX_CO27", "A horse pulling a cart",     ["horse"],       {"horse":1},         False,
     "Action+object. cart not in COCO."),
    ("EX_CO28", "A chef cooking in a kitchen",["person"],      {"person":1},        False,
     "Role+action. CLIP handles kitchen."),
    ("EX_CO29", "A boat sailing on water",    ["boat"],        {"boat":1},          False,
     "Object+action+scene. YOLO: boat=1."),
    ("EX_CO30", "A plane flying over mountains", ["airplane"], {"airplane":1},      False,
     "Object+action+scene. YOLO: airplane=1."),
    # Complex multi-attribute compositional
    ("EX_CO31", "A red chair beside a blue table", [], {}, True,
     "Two coloured objects. CLIP-only."),
    ("EX_CO32", "A small dog on a large sofa", ["dog","couch"], {"dog":1,"couch":1}, False,
     "Size+spatial. Both in COCO."),
    ("EX_CO33", "A white plate with red strawberries", [], {}, True,
     "Colour+compositional. CLIP-only."),
    ("EX_CO34", "A glass of orange juice on a wooden table", [], {}, True,
     "Material+colour+spatial. CLIP-only."),
    ("EX_CO35", "A black cat sitting in a red box", ["cat"], {"cat":1}, False,
     "Colour+spatial. cat in COCO, box not."),
    ("EX_CO36", "A person in a blue jacket walking a dog", ["person","dog"], {"person":1,"dog":1}, False,
     "Colour+action+two objects."),
    ("EX_CO37", "A yellow rubber duck in a bathtub", [], {}, True,
     "Colour+spatial. CLIP-only."),
    ("EX_CO38", "A silver fork and knife on a white plate", [], {}, True,
     "Material+spatial. CLIP-only."),
    ("EX_CO39", "A teddy bear on a striped bed", [], {}, True,
     "Object+spatial+pattern. CLIP-only."),
    ("EX_CO40", "A laptop on a wooden desk next to a coffee cup", ["laptop","cup"], {"laptop":1,"cup":1}, False,
     "Two objects+spatial+material."),
]

for (pid, prompt, ye, yc, manual, note) in co_prompts:
    external_prompts.append({
        "id": pid, "category": "Compositional",
        "prompt": prompt,
        "yolo_expected": ye, "yolo_counts": yc,
        "yolo_forbidden": [], "manual": manual, "note": note
    })

# ════════════════════════════════════════════════════════
# CATEGORY 3: Negation (40 prompts)
# Tests SD's ability to respect negative constraints
# ════════════════════════════════════════════════════════
ne_prompts = [
    # Object-level negation (YOLO where main object in COCO)
    ("EX_NE01", "A car with no windows",    ["car"],    {"car":1},    False,
     "Negation: no windows. YOLO: car=1. window not in COCO."),
    ("EX_NE02", "A bicycle with no seat",   ["bicycle"],{"bicycle":1},False,
     "Negation: no seat. YOLO: bicycle=1."),
    ("EX_NE03", "A dog with no tail",       ["dog"],    {"dog":1},    False,
     "Negation: no tail. YOLO: dog=1."),
    ("EX_NE04", "A cat with no ears",       ["cat"],    {"cat":1},    False,
     "Negation: no ears. YOLO: cat=1."),
    ("EX_NE05", "A person with no shadow",  ["person"], {"person":1}, False,
     "Negation: no shadow. YOLO: person=1."),
    ("EX_NE06", "A boat with no sail",      ["boat"],   {"boat":1},   False,
     "Negation: no sail. YOLO: boat=1."),
    ("EX_NE07", "An airplane with no wings",["airplane"],{"airplane":1},False,
     "Negation: no wings. YOLO: airplane=1. Critical structural negation."),
    ("EX_NE08", "A horse with no legs",     ["horse"],  {"horse":1},  False,
     "Negation: no legs. YOLO: horse=1."),
    ("EX_NE09", "A clock with no hands",    [], {}, True,
     "Negation: no hands. clock not in COCO. CLIP-only."),
    ("EX_NE10", "A chair with no legs",     ["chair"],  {"chair":1},  False,
     "Negation: no legs. YOLO: chair=1."),
    # Attribute negation
    ("EX_NE11", "A red apple with no stem", ["apple"],  {"apple":1},  False,
     "Attribute negation. YOLO: apple=1."),
    ("EX_NE12", "A tree with no branches",  [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE13", "A building with no windows", [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE14", "A flower with no petals",  [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE15", "A book with no cover",     [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE16", "A guitar with no strings", [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE17", "A bird with no feathers",  ["bird"],   {"bird":1},   False,
     "Attribute negation. YOLO: bird=1."),
    ("EX_NE18", "A sun with no rays",       [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE19", "A cup with no handle",     ["cup"],    {"cup":1},    False,
     "Attribute negation. YOLO: cup=1."),
    ("EX_NE20", "A table with no surface",  ["dining table"], {"dining table":1}, False,
     "Attribute negation. YOLO: dining table=1."),
    # Scene-level negation
    ("EX_NE21", "A beach with no water",    [], {}, True,
     "Scene negation. CLIP-only."),
    ("EX_NE22", "A sky with no clouds",     [], {}, True,
     "Scene negation. CLIP-only."),
    ("EX_NE23", "A forest with no trees",   [], {}, True,
     "Scene negation — paradoxical. CLIP-only."),
    ("EX_NE24", "A city with no people",    ["person"], {}, False,
     "Scene negation. YOLO checks person absence (yolo_counts empty = presence check, but forbidden expected)."),
    ("EX_NE25", "A kitchen with no food",   [], {}, True,
     "Scene negation. CLIP-only."),
    ("EX_NE26", "A road with no cars",      ["car"],    {}, False,
     "Scene negation. YOLO: car should be absent — presence check inverted."),
    ("EX_NE27", "A library with no books",  [], {}, True,
     "Scene negation. CLIP-only."),
    ("EX_NE28", "A swimming pool with no water", [], {}, True,
     "Scene negation. CLIP-only."),
    ("EX_NE29", "A playground with no children", ["person"], {}, False,
     "Scene negation. YOLO presence-inverted."),
    ("EX_NE30", "A parking lot with no cars", ["car"], {}, False,
     "Scene negation. YOLO presence-inverted."),
    # Colour negation
    ("EX_NE31", "A banana that is not yellow", ["banana"], {"banana":1}, False,
     "Colour negation. YOLO: banana=1. CLIP checks non-yellow."),
    ("EX_NE32", "A strawberry that is not red", [], {}, True,
     "Colour negation. CLIP-only."),
    ("EX_NE33", "Grass that is not green",  [], {}, True,
     "Colour negation. CLIP-only."),
    ("EX_NE34", "A sky that is not blue",   [], {}, True,
     "Colour negation. CLIP-only."),
    ("EX_NE35", "Snow that is not white",   [], {}, True,
     "Colour negation. CLIP-only."),
    # Combined negation
    ("EX_NE36", "A person with no face",    ["person"], {"person":1}, False,
     "Body-part negation. YOLO: person=1. CLIP checks face absence."),
    ("EX_NE37", "A hand with no fingers",   [], {}, True,
     "Body-part negation. CLIP-only."),
    ("EX_NE38", "A face with no mouth",     ["person"], {"person":1}, False,
     "Body-part negation. YOLO: person=1."),
    ("EX_NE39", "A fish with no fins",      [], {}, True,
     "Attribute negation. CLIP-only."),
    ("EX_NE40", "A door with no handle",    [], {}, True,
     "Attribute negation. CLIP-only."),
]

for (pid, prompt, ye, yc, manual, note) in ne_prompts:
    external_prompts.append({
        "id": pid, "category": "Negation",
        "prompt": prompt,
        "yolo_expected": ye, "yolo_counts": yc,
        "yolo_forbidden": [], "manual": manual, "note": note
    })

# ════════════════════════════════════════════════════════
# CATEGORY 4: Ambiguous (40 prompts)
# Underspecified prompts — model must resolve ambiguity
# ════════════════════════════════════════════════════════
am_prompts = [
    # Underspecified category
    ("EX_AM01", "A bird",          ["bird"],{"bird":1},False, "Species ambiguous."),
    ("EX_AM02", "A flower",        [], {}, True, "Species ambiguous. CLIP-only."),
    ("EX_AM03", "A tree",          [], {}, True, "Species ambiguous. CLIP-only."),
    ("EX_AM04", "A building",      [], {}, True, "Type ambiguous. CLIP-only."),
    ("EX_AM05", "A food",          [], {}, True, "Category ambiguous. CLIP-only."),
    ("EX_AM06", "A drink",         [], {}, True, "Category ambiguous. CLIP-only."),
    ("EX_AM07", "A sport",         [], {}, True, "Category ambiguous. CLIP-only."),
    ("EX_AM08", "A landscape",     [], {}, True, "Category ambiguous. CLIP-only."),
    ("EX_AM09", "A face",          ["person"],{"person":1},False, "Identity ambiguous. YOLO: person=1."),
    ("EX_AM10", "A pet",           ["cat","dog"],{},False, "Species ambiguous. YOLO: cat or dog present."),
    # Ambiguous count
    ("EX_AM11", "Some apples",     ["apple"],{},False, "Count ambiguous. YOLO: apple present."),
    ("EX_AM12", "Some people",     ["person"],{},False, "Count ambiguous. YOLO: person present."),
    ("EX_AM13", "Several birds",   ["bird"],{},False, "Count ambiguous. YOLO: bird present."),
    ("EX_AM14", "A few cars",      ["car"],{},False, "Count ambiguous. YOLO: car present."),
    ("EX_AM15", "Many flowers",    [], {}, True, "Count ambiguous. CLIP-only."),
    # Ambiguous scene
    ("EX_AM16", "A nature scene",  [], {}, True, "Scene ambiguous. CLIP-only."),
    ("EX_AM17", "An urban scene",  [], {}, True, "Scene ambiguous. CLIP-only."),
    ("EX_AM18", "An indoor scene", [], {}, True, "Scene ambiguous. CLIP-only."),
    ("EX_AM19", "A winter scene",  [], {}, True, "Scene ambiguous. CLIP-only."),
    ("EX_AM20", "A summer scene",  [], {}, True, "Scene ambiguous. CLIP-only."),
    # Ambiguous colour
    ("EX_AM21", "A colourful object", [], {}, True, "Colour ambiguous. CLIP-only."),
    ("EX_AM22", "A dark room",     [], {}, True, "Scene+colour ambiguous. CLIP-only."),
    ("EX_AM23", "A bright light",  [], {}, True, "Object ambiguous. CLIP-only."),
    ("EX_AM24", "Something red",   [], {}, True, "Object completely ambiguous. CLIP-only."),
    ("EX_AM25", "Something round", [], {}, True, "Object completely ambiguous. CLIP-only."),
    # Ambiguous action/state
    ("EX_AM26", "Something running",["person","dog","cat"],{},False,"Action ambiguous. YOLO: any mover."),
    ("EX_AM27", "Something flying", ["bird","airplane","kite"],{},False,"Action ambiguous. YOLO: any flier."),
    ("EX_AM28", "Something swimming",[], {}, True, "Action ambiguous. CLIP-only."),
    ("EX_AM29", "Something sleeping",["person","dog","cat"],{},False,"Action ambiguous. YOLO: any sleeper."),
    ("EX_AM30", "Something growing", [], {}, True, "Action ambiguous. CLIP-only."),
    # Ambiguous size
    ("EX_AM31", "A large animal",  ["cow","horse","elephant"],{},False,"Size+species ambiguous."),
    ("EX_AM32", "A small creature",[], {}, True, "Size+species ambiguous. CLIP-only."),
    ("EX_AM33", "A tiny thing",    [], {}, True, "Size completely ambiguous. CLIP-only."),
    ("EX_AM34", "A huge structure",[], {}, True, "Size+type ambiguous. CLIP-only."),
    ("EX_AM35", "Something old",   [], {}, True, "State ambiguous. CLIP-only."),
    # Conceptually ambiguous
    ("EX_AM36", "A musical sound", [], {}, True, "Multi-modal ambiguity. CLIP-only."),
    ("EX_AM37", "A warm place",    [], {}, True, "Conceptual ambiguity. CLIP-only."),
    ("EX_AM38", "A safe space",    [], {}, True, "Conceptual ambiguity. CLIP-only."),
    ("EX_AM39", "An interesting object", [], {}, True, "Conceptual ambiguity. CLIP-only."),
    ("EX_AM40", "A beautiful thing", [], {}, True, "Conceptual ambiguity. CLIP-only."),
]

for (pid, prompt, ye, yc, manual, note) in am_prompts:
    external_prompts.append({
        "id": pid, "category": "Ambiguous",
        "prompt": prompt,
        "yolo_expected": ye, "yolo_counts": yc,
        "yolo_forbidden": [], "manual": manual, "note": note
    })

# ════════════════════════════════════════════════════════
# CATEGORY 5: Conflicting (40 prompts — 20 impossible, 20 near-impossible)
# Tests geometric oscillation under semantic contradiction
# ════════════════════════════════════════════════════════
cf_prompts = [
    # Physically impossible (directly contradictory attributes)
    ("EX_CF01", "A dry ocean",                 [], {}, True, "Contradictory: ocean requires water."),
    ("EX_CF02", "A dark light",                [], {}, True, "Oxymoron: light cannot be dark."),
    ("EX_CF03", "A cold fire",                 [], {}, True, "Contradictory temperature."),
    ("EX_CF04", "A silent explosion",          [], {}, True, "Contradictory sensory properties."),
    ("EX_CF05", "A weightless rock",           [], {}, True, "Contradictory physical property."),
    ("EX_CF06", "An invisible painting",       [], {}, True, "Contradictory visibility."),
    ("EX_CF07", "A liquid wall",               [], {}, True, "Contradictory state of matter."),
    ("EX_CF08", "A solid rainbow",             [], {}, True, "Contradictory physical form."),
    ("EX_CF09", "A burning ice cube",          [], {}, True, "Contradictory temperature/state."),
    ("EX_CF10", "A hollow mountain",           [], {}, True, "Contradictory structure."),
    ("EX_CF11", "An edible metal",             [], {}, True, "Contradictory material properties."),
    ("EX_CF12", "A sleeping explosion",        [], {}, True, "Contradictory state."),
    ("EX_CF13", "A loud whisper",              [], {}, True, "Contradictory sound properties."),
    ("EX_CF14", "A sharp cloud",               [], {}, True, "Contradictory physical properties."),
    ("EX_CF15", "A heavy feather",             [], {}, True, "Contradictory weight."),
    ("EX_CF16", "A colourless rainbow",        [], {}, True, "Contradictory definition."),
    ("EX_CF17", "An opaque window",            [], {}, True, "Contradictory transparency."),
    ("EX_CF18", "A blind eye that sees",       [], {}, True, "Contradictory function."),
    ("EX_CF19", "A stationary river",          [], {}, True, "Contradictory defining property."),
    ("EX_CF20", "A round triangle",            [], {}, True, "Geometric impossibility."),
    # Near-impossible (deep semantic tension)
    ("EX_CF21", "Frozen smoke",                [], {}, True, "Near-impossible state."),
    ("EX_CF22", "Living stone",                [], {}, True, "Contradictory life/material."),
    ("EX_CF23", "Transparent shadow",          [], {}, True, "Contradictory opacity."),
    ("EX_CF24", "Musical colour",              [], {}, True, "Cross-modal contradiction."),
    ("EX_CF25", "Tasting a thought",           [], {}, True, "Cross-modal contradiction."),
    ("EX_CF26", "A square wave",               [], {}, True, "Mathematical impossibility."),
    ("EX_CF27", "Glass made of water",         [], {}, True, "Material contradiction."),
    ("EX_CF28", "A mirror that does not reflect", [], {}, True, "Functional contradiction."),
    ("EX_CF29", "A clock running backwards",   [], {}, True, "Functional contradiction."),
    ("EX_CF30", "A black white photograph",    [], {}, True, "Colour contradiction."),
    ("EX_CF31", "A living skeleton",           [], {}, True, "State contradiction."),
    ("EX_CF32", "An empty container full of nothing", [], {}, True, "Semantic tautology/contradiction."),
    ("EX_CF33", "A two-sided coin with one face", [], {}, True, "Mathematical impossibility."),
    ("EX_CF34", "A beginning with no end",     [], {}, True, "Conceptual contradiction."),
    ("EX_CF35", "A speaking silence",          [], {}, True, "Contradiction."),
    ("EX_CF36", "Warm darkness",               [], {}, True, "Cross-modal tension."),
    ("EX_CF37", "A concrete dream",            [], {}, True, "Abstract/concrete contradiction."),
    ("EX_CF38", "An ancient newborn",          [], {}, True, "Temporal contradiction."),
    ("EX_CF39", "A broken whole",              [], {}, True, "State contradiction."),
    ("EX_CF40", "The sound of emptiness",      [], {}, True, "Cross-modal impossibility."),
]

for (pid, prompt, ye, yc, manual, note) in cf_prompts:
    external_prompts.append({
        "id": pid, "category": "Conflicting",
        "prompt": prompt,
        "yolo_expected": ye, "yolo_counts": yc,
        "yolo_forbidden": [], "manual": manual, "note": note
    })

# ── Summary ───────────────────────────────────────────────────────────────────
from collections import Counter
cat_counts = Counter(p["category"] for p in external_prompts)
manual_count = sum(1 for p in external_prompts if p["manual"])

print(f"Total external prompts: {len(external_prompts)}")
print(f"\nPer category:")
for cat, n in cat_counts.items():
    print(f"  {cat:<20} {n}")
print(f"\nManual (CLIP-only): {manual_count}")
print(f"AUTO (YOLO+CLIP):   {len(external_prompts) - manual_count}")

Total external prompts: 200

Per category:
  Single-Concept       40
  Compositional        40
  Negation             40
  Ambiguous            40
  Conflicting          40

Manual (CLIP-only): 129
AUTO (YOLO+CLIP):   71


In [ ]:
out_path = "Phase8_Data/prompts/prompt_config_ext.json"
with open(out_path, "w") as f:
    json.dump(external_prompts, f, indent=2)

print(f"Saved: {out_path}")
print(f"File size: {os.path.getsize(out_path)/1024:.1f} KB")

with open(out_path) as f:
    verify = json.load(f)
assert len(verify) == len(external_prompts), "Count mismatch!"
assert all(k in verify[0] for k in ["id","category","prompt","yolo_expected",
                                     "yolo_counts","yolo_forbidden","manual","note"])
print("Verification passed. Format identical to Phase 3 prompt_config.json.")
print("\nFirst entry:")
print(json.dumps(verify[0], indent=2))
print("\nLast entry:")
print(json.dumps(verify[-1], indent=2))

Saved: Phase8_Data/prompts/prompt_config_ext.json
File size: 52.1 KB
Verification passed. Format identical to Phase 3 prompt_config.json.

First entry:
{
  "id": "EX_SC01",
  "category": "Single-Concept",
  "prompt": "A red fire truck",
  "yolo_expected": [
    "truck"
  ],
  "yolo_counts": {
    "truck": 1
  },
  "yolo_forbidden": [],
  "manual": false,
  "note": "Color+object. YOLO: truck=1."
}

Last entry:
{
  "id": "EX_CF40",
  "category": "Conflicting",
  "prompt": "The sound of emptiness",
  "yolo_expected": [],
  "yolo_counts": {},
  "yolo_forbidden": [],
  "manual": true,
  "note": "Cross-modal impossibility."
}


In [ ]:
import os
import json

out_path = "Phase8_Data/prompts/prompt_config_ext.json"

IMG_DIR   = "Phase8_Data/generated/"
TRAJ_DIR  = "Phase8_Data/trajectories/"
PROMPT_FILE = "Phase8_Data/prompts/prompt_config_ext.json"

img_files  = [f for f in os.listdir(IMG_DIR)  if f.endswith(".png")]
traj_files = [f for f in os.listdir(TRAJ_DIR) if f.endswith(".npy")]

print(f"Images found:       {len(img_files)}")
print(f"Trajectories found: {len(traj_files)}")
print(f"Expected:           10,000 each")
print(f"Match: {len(img_files) == len(traj_files)}")
print(f"\nExample image: {sorted(img_files)[0]}")
print(f"Example traj:  {sorted(traj_files)[0]}")

# Check one trajectory shape
import numpy as np
sample = np.load(os.path.join(TRAJ_DIR, sorted(traj_files)[0]))
print(f"\nTrajectory shape: {sample.shape}")   # should be (15, 4, 64, 64)
print(f"Dtype: {sample.dtype}")               # should be float32

Images found:       10000
Trajectories found: 10000
Expected:           10,000 each
Match: True

Example image: experimental__EX_AM01__s000__cfg7.5.png
Example traj:  experimental__EX_AM01__s000__cfg7.5.npy

Trajectory shape: (15, 4, 64, 64)
Dtype: float32


In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm

IMG_DIR    = "Phase8_Data/generated/"
TRAJ_DIR   = "Phase8_Data/trajectories/"
PROMPT_FILE = "Phase8_Data/prompts/prompt_config_ext.json"
OUT_DIR    = "Phase8_Results/"
os.makedirs(OUT_DIR, exist_ok=True)

with open(PROMPT_FILE) as f:
    prompts = json.load(f)
prompt_lookup = {p["id"]: p for p in prompts}


with open("Phase4_Results/thresholds.json") as f:
    thresholds = json.load(f)

print(f"Prompts loaded:    {len(prompts)}")
print(f"Thresholds loaded: {list(thresholds.keys())}")

traj_files = sorted([f for f in os.listdir(TRAJ_DIR) if f.endswith(".npy")])
metadata   = []

for fname in tqdm(traj_files, desc="Building metadata"):
    key = fname.replace(".npy", "")


    parts     = key.split("__")
    prompt_id = parts[1]
    seed      = int(parts[2].replace("s", ""))
    cfg_val   = float(parts[3].replace("cfg", ""))

    p        = prompt_lookup[prompt_id]
    traj     = np.load(os.path.join(TRAJ_DIR, fname))
    variance = float(traj.var(axis=0).mean())

    metadata.append({
        "key"      : key,
        "split"    : "experimental",
        "prompt_id": prompt_id,
        "category" : p["category"],
        "prompt"   : p["prompt"],
        "seed"     : seed,
        "cfg"      : cfg_val,
        "variance" : variance,
        "manual"   : p["manual"],
    })

meta_path = "Phase8_Data/phase8_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nMetadata built: {len(metadata)} records")
print(f"Saved: {meta_path}")

vars_all = [m["variance"] for m in metadata]
print(f"\nVariance distribution:")
print(f"  min:    {min(vars_all):.6f}")
print(f"  max:    {max(vars_all):.6f}")
print(f"  mean:   {np.mean(vars_all):.6f}")
print(f"  median: {np.median(vars_all):.6f}")
print(f"\nPhase 3 range was: 0.000691 to 0.018349")
print(f"Ranges should be similar ↑")

Prompts loaded:    200
Thresholds loaded: ['Ambiguous', 'Compositional', 'Conflicting', 'Negation', 'Single-Concept', '__global__']


Building metadata: 100%|██████████| 10000/10000 [04:49<00:00, 34.56it/s]



Metadata built: 10000 records
Saved: Phase8_Data/phase8_metadata.json

Variance distribution:
  min:    0.000349
  max:    0.008368
  mean:   0.002754
  median: 0.002748

Phase 3 range was: 0.000691 to 0.018349
Ranges should be similar ↑


In [ ]:
import open_clip
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-L-14", pretrained="openai"
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer("ViT-L-14")
print("CLIP model loaded: ViT-L-14 (OpenAI)")

@torch.no_grad()
def clip_score(image_path: str, prompt_text: str) -> float:
    img  = clip_preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    txt  = clip_tokenizer([prompt_text]).to(device)
    img_feat = clip_model.encode_image(img)
    txt_feat = clip_model.encode_text(txt)
    img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True)
    txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True)
    return float((img_feat * txt_feat).sum())

prompt_text_lookup = {p["id"]: p["prompt"] for p in prompts}

clip_scores = {}
for rec in tqdm(metadata, desc="CLIP scoring"):
    img_path = os.path.join(IMG_DIR, rec["key"] + ".png")
    score = clip_score(img_path, prompt_text_lookup[rec["prompt_id"]])
    clip_scores[rec["key"]] = score

with open("Phase8_Results/clip_scores.json", "w") as f:
    json.dump(clip_scores, f, indent=2)

scores = list(clip_scores.values())
print(f"\nCLIP scoring complete: {len(clip_scores)} images")
print(f"  min:    {min(scores):.4f}")
print(f"  max:    {max(scores):.4f}")
print(f"  mean:   {np.mean(scores):.4f}")
print(f"\nPhase 3 mean at CFG=7.5 was: 0.2401")
print(f"Should be similar ↑")

Device: cuda


c:\Users\Hp\AppData\Local\Programs\Python\Python310\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


CLIP model loaded: ViT-L-14 (OpenAI)


CLIP scoring: 100%|██████████| 10000/10000 [27:25<00:00,  6.08it/s]


CLIP scoring complete: 10000 images
  min:    0.1275
  max:    0.3761
  mean:   0.2406

Phase 3 mean at CFG=7.5 was: 0.2401
Should be similar ↑


In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8x.pt")
print("YOLOv8x loaded")

def run_yolo(image_path: str, prompt_cfg: dict) -> dict:
    """
    Run YOLO on one image. Returns:
        yolo_error : bool   — True if YOLO detects a violation
        yolo_detail: str    — description of what failed
    """
  
    if prompt_cfg["manual"]:
        return {"yolo_error": False, "yolo_detail": "manual_skip"}

    results  = yolo_model(image_path, verbose=False, conf=0.25)[0]
    detected = [yolo_model.names[int(c)] for c in results.boxes.cls]

    expected  = prompt_cfg["yolo_expected"]
    counts    = prompt_cfg["yolo_counts"]    # {} = presence-only
    forbidden = prompt_cfg["yolo_forbidden"]


    for obj in forbidden:
        if obj in detected:
            return {"yolo_error": True,
                    "yolo_detail": f"forbidden '{obj}' detected"}

    for obj in expected:
        if obj not in detected:
            return {"yolo_error": True,
                    "yolo_detail": f"expected '{obj}' not found"}

    for obj, expected_count in counts.items():
        actual_count = detected.count(obj)
        if actual_count != expected_count:
            return {"yolo_error": True,
                    "yolo_detail": f"'{obj}' count {actual_count} != {expected_count}"}

    return {"yolo_error": False, "yolo_detail": "pass"}

prompt_cfg_lookup = {p["id"]: p for p in prompts}
yolo_results = {}

for rec in tqdm(metadata, desc="YOLO scoring"):
    img_path   = os.path.join(IMG_DIR, rec["key"] + ".png")
    prompt_cfg = prompt_cfg_lookup[rec["prompt_id"]]
    result     = run_yolo(img_path, prompt_cfg)
    yolo_results[rec["key"]] = result

with open("Phase8_Results/yolo_results.json", "w") as f:
    json.dump(yolo_results, f, indent=2)

total_errors = sum(1 for v in yolo_results.values() if v["yolo_error"])
total_manual = sum(1 for v in yolo_results.values() if v["yolo_detail"] == "manual_skip")
total_auto   = len(yolo_results) - total_manual

print(f"\nYOLO scoring complete: {len(yolo_results)} images")
print(f"  AUTO prompts scored: {total_auto}")
print(f"  MANUAL (skipped):    {total_manual}")
print(f"  YOLO errors:         {total_errors} ({total_errors/total_auto*100:.1f}% of AUTO)")

YOLOv8x loaded


YOLO scoring: 100%|██████████| 10000/10000 [04:28<00:00, 37.31it/s] 


YOLO scoring complete: 10000 images
  AUTO prompts scored: 3550
  MANUAL (skipped):    6450
  YOLO errors:         1167 (32.9% of AUTO)


In [ ]:
phase8_results = []

for rec in metadata:
    key      = rec["key"]
    cat      = rec["category"]
    variance = rec["variance"]
    clip_sc  = clip_scores[key]
    yolo_res = yolo_results[key]

    T_var  = thresholds[cat]["T_var"]
    T_clip = thresholds[cat]["T_clip"]

    high_variance = variance > T_var
    low_clip      = clip_sc  < T_clip
    yolo_error    = yolo_res["yolo_error"]

    quality_fail  = yolo_error or low_clip

    if high_variance and quality_fail:
        failure_type = "both"
    elif high_variance and not quality_fail:
        failure_type = "geometric"
    elif not high_variance and quality_fail:
        failure_type = "semantic"
    else:
        failure_type = "none"

    phase8_results.append({
        "key"          : key,
        "prompt_id"    : rec["prompt_id"],
        "category"     : cat,
        "prompt"       : rec["prompt"],
        "seed"         : rec["seed"],
        "cfg"          : rec["cfg"],
        "variance"     : variance,
        "clip_score"   : clip_sc,
        "yolo_error"   : yolo_error,
        "yolo_detail"  : yolo_res["yolo_detail"],
        "T_var"        : T_var,
        "T_clip"       : T_clip,
        "high_variance": high_variance,
        "low_clip"     : low_clip,
        "quality_fail" : quality_fail,
        "failure_type" : failure_type,
        "hallucinated" : failure_type == "both",
    })

with open("Phase8_Results/phase8_labels.json", "w") as f:
    json.dump(phase8_results, f, indent=2)

print(f"Labels saved: {len(phase8_results)} records")

Labels saved: 10000 records


In [8]:
from collections import Counter

ft_counts = Counter(r["failure_type"] for r in phase8_results)
total     = len(phase8_results)

print("=" * 55)
print("PHASE 8C — HALLUCINATION LABEL SUMMARY")
print("=" * 55)

print(f"\nOverall distribution (n={total}):")
for ft in ["both", "geometric", "semantic", "none"]:
    n   = ft_counts[ft]
    pct = n / total * 100
    print(f"  {ft:<12}: {n:>5}  ({pct:.1f}%)")

hall_rate = ft_counts["both"] / total * 100
print(f"\nHallucination rate (both): {hall_rate:.2f}%")
print(f"Phase 3 rate was:          8.2%")

print(f"\nPer-category breakdown:")
cats = ["Single-Concept","Compositional","Negation","Ambiguous","Conflicting"]
print(f"  {'Category':<20} {'n':>5} {'Hall.':>6} {'Rate':>7} "
      f"{'Semantic':>9} {'Geometric':>10} {'Clean':>7}")
print("  " + "─"*65)

for cat in cats:
    cat_recs = [r for r in phase8_results if r["category"] == cat]
    n_total  = len(cat_recs)
    n_hall   = sum(1 for r in cat_recs if r["failure_type"] == "both")
    n_sem    = sum(1 for r in cat_recs if r["failure_type"] == "semantic")
    n_geo    = sum(1 for r in cat_recs if r["failure_type"] == "geometric")
    n_clean  = sum(1 for r in cat_recs if r["failure_type"] == "none")
    rate     = n_hall / n_total * 100 if n_total > 0 else 0
    print(f"  {cat:<20} {n_total:>5} {n_hall:>6} {rate:>6.1f}% "
          f"{n_sem:>9} {n_geo:>10} {n_clean:>7}")

print(f"\nVariance stats:")
p8_vars = [r["variance"] for r in phase8_results]
print(f"  mean:  {np.mean(p8_vars):.6f}  (Phase 3: 0.002732 at CFG=7.5)")
print(f"  std:   {np.std(p8_vars):.6f}")
print(f"  min:   {min(p8_vars):.6f}")
print(f"  max:   {max(p8_vars):.6f}")

print(f"\nCLIP stats:")
p8_clips = [r["clip_score"] for r in phase8_results]
print(f"  mean:  {np.mean(p8_clips):.4f}  (Phase 3: 0.2401 at CFG=7.5)")
print(f"  std:   {np.std(p8_clips):.4f}")

PHASE 8C — HALLUCINATION LABEL SUMMARY

Overall distribution (n=10000):
  both        :    97  (1.0%)
  geometric   :   352  (3.5%)
  semantic    :  1500  (15.0%)
  none        :  8051  (80.5%)

Hallucination rate (both): 0.97%
Phase 3 rate was:          8.2%

Per-category breakdown:
  Category                 n  Hall.    Rate  Semantic  Geometric   Clean
  ─────────────────────────────────────────────────────────────────
  Single-Concept        2000      7    0.4%       116        213    1664
  Compositional         2000     73    3.6%       473         90    1364
  Negation              2000     15    0.8%       350         44    1591
  Ambiguous             2000      1    0.1%       486          3    1510
  Conflicting           2000      1    0.1%        75          2    1922

Variance stats:
  mean:  0.002754  (Phase 3: 0.002732 at CFG=7.5)
  std:   0.000626
  min:   0.000349
  max:   0.008368

CLIP stats:
  mean:  0.2406  (Phase 3: 0.2401 at CFG=7.5)
  std:   0.0320


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from scipy import stats
import numpy as np
import math

y_p8       = np.array([1 if r["hallucinated"] else 0 for r in phase8_results])
X_var_p8   = np.array([r["variance"]   for r in phase8_results])
X_clip_p8  = np.array([r["clip_score"] for r in phase8_results])

print(f"Phase 8 label distribution:")
print(f"  Hallucinated: {y_p8.sum()} ({y_p8.mean()*100:.2f}%)")
print(f"  Clean:        {(1-y_p8).sum()} ({(1-y_p8).mean()*100:.2f}%)")

auc_var_p8  = roc_auc_score(y_p8,  X_var_p8)
auc_clip_p8 = roc_auc_score(y_p8, -X_clip_p8)  # negate: lower clip = more hallucinated

scaler_p8 = StandardScaler()
X_comb_p8 = scaler_p8.fit_transform(np.column_stack([X_var_p8, X_clip_p8]))
lr_p8     = LogisticRegression(random_state=42, max_iter=1000)
cv_p8     = cross_val_score(lr_p8, X_comb_p8, y_p8, cv=5, scoring='roc_auc')
lr_p8.fit(X_comb_p8, y_p8)
auc_comb_p8 = roc_auc_score(y_p8, lr_p8.predict_proba(X_comb_p8)[:,1])

with open("Phase4_Results/phase4_results.json") as f:
    p3_results = json.load(f)

y_p3      = np.array([1 if r["hallucinated"] else 0 for r in p3_results])
X_var_p3  = np.array([r["variance"]   for r in p3_results])
X_clip_p3 = np.array([r["clip_score"] for r in p3_results])

auc_var_p3  = roc_auc_score(y_p3,  X_var_p3)
auc_clip_p3 = roc_auc_score(y_p3, -X_clip_p3)

scaler_p3 = StandardScaler()
X_comb_p3 = scaler_p3.fit_transform(np.column_stack([X_var_p3, X_clip_p3]))
lr_p3     = LogisticRegression(random_state=42, max_iter=1000)
lr_p3.fit(X_comb_p3, y_p3)
auc_comb_p3 = roc_auc_score(y_p3, lr_p3.predict_proba(X_comb_p3)[:,1])

print(f"\n{'='*65}")
print(f"PHASE 8D — GENERALISATION AUC COMPARISON")
print(f"{'='*65}")
print(f"\n{'Signal':<30} {'Internal (Ph3)':>15} {'External (Ph8)':>15} {'Generalised?':>14}")
print("─"*65)

results_table = [
    ("Global Variance alone",   auc_var_p3,  auc_var_p8),
    ("CLIP Score alone",        auc_clip_p3, auc_clip_p8),
    ("Combined (Var + CLIP)",   auc_comb_p3, auc_comb_p8),
]

for name, internal, external in results_table:
    drop  = internal - external
    threshold = 0.80 if "Combined" in name else 0.75
    passed = "✓ PASS" if external >= threshold else "✗ FAIL"
    print(f"  {name:<28} {internal:>15.4f} {external:>15.4f} {passed:>14}")

print(f"\n  5-fold CV (Phase 8 combined): {cv_p8.mean():.4f} ± {cv_p8.std():.4f}")
print(f"\n  Pass thresholds: Variance≥0.75, CLIP≥0.75, Combined≥0.80")

Phase 8 label distribution:
  Hallucinated: 97 (0.97%)
  Clean:        9903 (99.03%)

PHASE 8D — GENERALISATION AUC COMPARISON

Signal                          Internal (Ph3)  External (Ph8)   Generalised?
─────────────────────────────────────────────────────────────────
  Global Variance alone                 0.8766          0.9585         ✓ PASS
  CLIP Score alone                      0.8725          0.4751         ✗ FAIL
  Combined (Var + CLIP)                 0.8979          0.9604         ✓ PASS

  5-fold CV (Phase 8 combined): 0.9377 ± 0.0321

  Pass thresholds: Variance≥0.75, CLIP≥0.75, Combined≥0.80


In [10]:
cats_with_hall = [cat for cat in
                  ["Single-Concept","Compositional","Negation","Ambiguous","Conflicting"]
                  if sum(1 for r in phase8_results
                         if r["category"]==cat and r["hallucinated"]) >= 5]

print(f"Categories with ≥5 hallucinated images (AUC computable): {cats_with_hall}")
print(f"\n{'Category':<20} {'n_hall':>7} {'Var AUC':>9} {'CLIP AUC':>10} {'Comb AUC':>10}")
print("─"*60)

per_cat_aucs = {}
for cat in ["Single-Concept","Compositional","Negation","Ambiguous","Conflicting"]:
    cat_recs  = [r for r in phase8_results if r["category"] == cat]
    y_cat     = np.array([1 if r["hallucinated"] else 0 for r in cat_recs])
    n_hall    = y_cat.sum()

    if n_hall < 5:
        print(f"  {cat:<20} {n_hall:>7}   {'N/A':>9} {'N/A':>10} {'N/A':>10}  (too few hallucinations)")
        continue

    X_v  = np.array([r["variance"]   for r in cat_recs])
    X_c  = np.array([r["clip_score"] for r in cat_recs])

    auc_v = roc_auc_score(y_cat,  X_v)
    auc_c = roc_auc_score(y_cat, -X_c)

    sc    = StandardScaler()
    X_cm  = sc.fit_transform(np.column_stack([X_v, X_c]))
    lr_c  = LogisticRegression(random_state=42, max_iter=1000)
    lr_c.fit(X_cm, y_cat)
    auc_cm = roc_auc_score(y_cat, lr_c.predict_proba(X_cm)[:,1])

    per_cat_aucs[cat] = {"var": auc_v, "clip": auc_c, "combined": auc_cm}
    print(f"  {cat:<20} {n_hall:>7} {auc_v:>9.4f} {auc_c:>10.4f} {auc_cm:>10.4f}")

Categories with ≥5 hallucinated images (AUC computable): ['Single-Concept', 'Compositional', 'Negation']

Category              n_hall   Var AUC   CLIP AUC   Comb AUC
────────────────────────────────────────────────────────────
  Single-Concept             7    0.9597     0.6899     0.9366
  Compositional             73    0.9778     0.6020     0.9780
  Negation                  15    0.9858     0.7472     0.9791
  Ambiguous                  1         N/A        N/A        N/A  (too few hallucinations)
  Conflicting                1         N/A        N/A        N/A  (too few hallucinations)


In [ ]:
hall_var_p8  = [r["variance"] for r in phase8_results if r["hallucinated"]]
clean_var_p8 = [r["variance"] for r in phase8_results if not r["hallucinated"]]

print(f"Hallucinated (n={len(hall_var_p8)})  — variance mean: {np.mean(hall_var_p8):.6f}")
print(f"Clean        (n={len(clean_var_p8)}) — variance mean: {np.mean(clean_var_p8):.6f}")

mw_p8 = stats.mannwhitneyu(hall_var_p8, clean_var_p8, alternative='greater')

n_total = len(hall_var_p8) + len(clean_var_p8)
z_val   = abs(stats.norm.ppf(mw_p8.pvalue / 2)) if mw_p8.pvalue > 0 else 8.0
r_val   = z_val / math.sqrt(n_total)

def cohens_d(a, b):
    pooled = math.sqrt((np.std(a)**2 + np.std(b)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else 0

d_p8 = cohens_d(hall_var_p8, clean_var_p8)

print(f"\nMann-Whitney U = {mw_p8.statistic:.0f}")
print(f"p-value        = {mw_p8.pvalue:.3e}")
print(f"Effect size r  = {r_val:.3f}")
print(f"Cohen's d      = {d_p8:.4f}")
print(f"\nPhase 3 comparison:")
print(f"  Phase 3 Cohen's d: 0.82    | Phase 8: {d_p8:.4f}")
print(f"  Phase 3 p-value:   2.70e-197 | Phase 8: {mw_p8.pvalue:.3e}")

Hallucinated (n=97)  — variance mean: 0.003994
Clean        (n=9903) — variance mean: 0.002742

Mann-Whitney U = 920685
p-value        = 6.340e-55
Effect size r  = 0.156
Cohen's d      = 1.8791

Phase 3 comparison:
  Phase 3 Cohen's d: 0.82    | Phase 8: 1.8791
  Phase 3 p-value:   2.70e-197 | Phase 8: 6.340e-55


In [ ]:
phase8_summary = {
    "dataset": {
        "n_images"        : 10000,
        "n_prompts"       : 200,
        "n_seeds"         : 50,
        "cfg"             : 7.5,
        "hallucinated"    : int(y_p8.sum()),
        "hall_rate_pct"   : float(y_p8.mean() * 100),
    },
    "variance_stats": {
        "mean"   : float(np.mean(X_var_p8)),
        "std"    : float(np.std(X_var_p8)),
        "min"    : float(X_var_p8.min()),
        "max"    : float(X_var_p8.max()),
        "phase3_mean_cfg7.5": 0.002732,
    },
    "clip_stats": {
        "mean"   : float(np.mean(X_clip_p8)),
        "std"    : float(np.std(X_clip_p8)),
        "phase3_mean_cfg7.5": 0.2401,
    },
    "auc_results": {
        "internal_variance_auc" : float(auc_var_p3),
        "internal_clip_auc"     : float(auc_clip_p3),
        "internal_combined_auc" : float(auc_comb_p3),
        "external_variance_auc" : float(auc_var_p8),
        "external_clip_auc"     : float(auc_clip_p8),
        "external_combined_auc" : float(auc_comb_p8),
        "external_combined_cv"  : f"{cv_p8.mean():.4f}±{cv_p8.std():.4f}",
        "variance_drop"         : float(auc_var_p3 - auc_var_p8),
        "combined_drop"         : float(auc_comb_p3 - auc_comb_p8),
    },
    "statistical_tests": {
        "mannwhitney_p"  : float(mw_p8.pvalue),
        "effect_r"       : float(r_val),
        "cohens_d"       : float(d_p8),
        "phase3_cohens_d": 0.82,
    },
    "per_category_auc": per_cat_aucs,
}

with open("Phase8_Results/phase8_summary.json", "w") as f:
    json.dump(phase8_summary, f, indent=2)

print("Phase 8 summary saved.")
print()

print("=" * 55)
print("GENERALISATION VERDICT")
print("=" * 55)

var_drop  = auc_var_p3  - auc_var_p8
comb_drop = auc_comb_p3 - auc_comb_p8

print(f"\nVariance AUC:  {auc_var_p3:.4f} → {auc_var_p8:.4f}  (drop: {var_drop:+.4f})")
print(f"Combined AUC:  {auc_comb_p3:.4f} → {auc_comb_p8:.4f}  (drop: {comb_drop:+.4f})")

print(f"\nInterpretation:")
if auc_var_p8 >= 0.75:
    print(f"  ✓ Variance AUC {auc_var_p8:.4f} ≥ 0.75 — GENERALISES to external prompts")
else:
    print(f"  ✗ Variance AUC {auc_var_p8:.4f} < 0.75 — does not generalise strongly")

if auc_comb_p8 >= 0.80:
    print(f"  ✓ Combined AUC {auc_comb_p8:.4f} ≥ 0.80 — GENERALISES to external prompts")
else:
    print(f"  ✗ Combined AUC {auc_comb_p8:.4f} < 0.80 — limited generalisation")

if var_drop <= 0.05:
    print(f"  ✓ AUC drop {var_drop:.4f} ≤ 0.05 — minimal distribution shift")
else:
    print(f"  ⚠ AUC drop {var_drop:.4f} > 0.05 — meaningful distribution shift detected")

print(f"\n  Note: 97 hallucinated / 10,000 images = 0.97%")
print(f"  Phase 3 at CFG=7.5 had 0/1000 = 0.0%")
print(f"  External prompts are harder — more diverse semantics")
print(f"  Variance and CLIP signals remain well-calibrated (means match Phase 3)")

Phase 8 summary saved.

GENERALISATION VERDICT

Variance AUC:  0.8766 → 0.9585  (drop: -0.0819)
Combined AUC:  0.8979 → 0.9604  (drop: -0.0625)

Interpretation:
  ✓ Variance AUC 0.9585 ≥ 0.75 — GENERALISES to external prompts
  ✓ Combined AUC 0.9604 ≥ 0.80 — GENERALISES to external prompts
  ✓ AUC drop -0.0819 ≤ 0.05 — minimal distribution shift

  Note: 97 hallucinated / 10,000 images = 0.97%
  Phase 3 at CFG=7.5 had 0/1000 = 0.0%
  External prompts are harder — more diverse semantics
  Variance and CLIP signals remain well-calibrated (means match Phase 3)


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.stats import gaussian_kde
import numpy as np

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#0f0f0f')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.32)

ax1 = fig.add_subplot(gs[0, 0])   # AUC comparison bar chart
ax2 = fig.add_subplot(gs[0, 1])   # ROC curves overlay
ax3 = fig.add_subplot(gs[0, 2])   # Variance distribution
ax4 = fig.add_subplot(gs[1, 0])   # Per-category AUC
ax5 = fig.add_subplot(gs[1, 1])   # CLIP failure explanation
ax6 = fig.add_subplot(gs[1, 2])   # Summary stats table

for ax in [ax1,ax2,ax3,ax4,ax5,ax6]:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

labels   = ['Variance\nalone', 'CLIP\nalone', 'Combined\n(Var+CLIP)']
internal = [auc_var_p3,  auc_clip_p3, auc_comb_p3]
external = [auc_var_p8,  auc_clip_p8, auc_comb_p8]

x     = np.arange(len(labels))
width = 0.35
b1 = ax1.bar(x - width/2, internal, width, label='Internal (Phase 3)',
             color='#4488ff', alpha=0.85)
b2 = ax1.bar(x + width/2, external, width, label='External (Phase 8)',
             color='#44cc88', alpha=0.85)

for bar in b1:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{bar.get_height():.3f}', ha='center', va='bottom',
             color='white', fontsize=8)
for bar in b2:
    col = '#ff4444' if bar.get_height() < 0.5 else 'white'
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{bar.get_height():.3f}', ha='center', va='bottom',
             color=col, fontsize=8)

ax1.axhline(0.75, color='#ffcc44', lw=1, ls='--', alpha=0.6, label='Pass threshold (0.75)')
ax1.axhline(0.50, color='#888888', lw=0.8, ls=':', alpha=0.5, label='Random baseline')
ax1.set_xticks(x); ax1.set_xticklabels(labels, color='white', fontsize=9)
ax1.set_ylim(0.3, 1.02)
ax1.set_ylabel('ROC-AUC', color='white', fontsize=10)
ax1.set_title('Internal vs External AUC\nVariance generalises; CLIP fails on diverse prompts',
              color='white', fontsize=9, fontweight='bold')
ax1.legend(fontsize=7.5, facecolor='#2a2a2a', labelcolor='white')

ax1.annotate('CLIP below\nrandom on\ndiverse prompts',
             xy=(1+width/2, auc_clip_p8),
             xytext=(1.6, 0.55),
             arrowprops=dict(arrowstyle='->', color='#ff4444', lw=1.5),
             color='#ff4444', fontsize=7.5, ha='center')


roc_configs = [
    ("Variance Internal", X_var_p3,  y_p3,  False, '#4488ff', '--', 1.5),
    ("Variance External", X_var_p8,  y_p8,  False, '#44cc88', '-',  2.5),
    ("CLIP Internal",     X_clip_p3, y_p3,  True,  '#aa66ff', '--', 1.5),
    ("CLIP External",     X_clip_p8, y_p8,  True,  '#ff4444', '-',  2.0),
]
for name, X, y, neg, color, ls, lw in roc_configs:
    X_use = -X if neg else X
    fpr, tpr, _ = roc_curve(y, X_use)
    auc_v = roc_auc_score(y, X_use)
    ax2.plot(fpr, tpr, color=color, lw=lw, ls=ls,
             label=f"{name} ({auc_v:.3f})")

ax2.plot([0,1],[0,1], color='#555555', lw=1, ls=':')
ax2.set_xlabel('False Positive Rate', color='white', fontsize=10)
ax2.set_ylabel('True Positive Rate',  color='white', fontsize=10)
ax2.set_title('ROC Curves: Internal vs External\nVariance stable; CLIP degrades',
              color='white', fontsize=9, fontweight='bold')
ax2.legend(fontsize=7.5, facecolor='#2a2a2a', labelcolor='white', loc='lower right')

hall_p8  = [r["variance"] for r in phase8_results if r["hallucinated"]]
clean_p8 = [r["variance"] for r in phase8_results if not r["hallucinated"]]
hall_p3  = [r["variance"] for r in p3_results if r["hallucinated"]]
clean_p3 = [r["variance"] for r in p3_results if not r["hallucinated"]]

x_range = np.linspace(0.0005, 0.010, 300)
for vals, label, color, ls in [
    (clean_p3, 'Clean Ph3',        '#4488ff', '--'),
    (hall_p3,  'Hallucinated Ph3', '#ff4444', '--'),
    (clean_p8, 'Clean Ph8',        '#44cc88', '-'),
    (hall_p8,  'Hallucinated Ph8', '#ffaa00', '-'),
]:
    if len(vals) < 5: continue
    kde = gaussian_kde(vals, bw_method=0.2)
    ax3.fill_between(x_range, kde(x_range), alpha=0.15, color=color)
    ax3.plot(x_range, kde(x_range), color=color, lw=2, ls=ls, label=label)

ax3.set_xlabel('Trajectory Variance', color='white', fontsize=10)
ax3.set_ylabel('Density', color='white', fontsize=10)
ax3.set_title(f'Variance Distribution: Ph3 vs Ph8\nCohen\'s d Ph8={d_p8:.2f} (Phase 3=0.82)',
              color='white', fontsize=9, fontweight='bold')
ax3.legend(fontsize=7.5, facecolor='#2a2a2a', labelcolor='white')


cat_labels = list(per_cat_aucs.keys())
var_aucs   = [per_cat_aucs[c]["var"]      for c in cat_labels]
clip_aucs  = [per_cat_aucs[c]["clip"]     for c in cat_labels]
comb_aucs  = [per_cat_aucs[c]["combined"] for c in cat_labels]

x4 = np.arange(len(cat_labels))
w4 = 0.25
ax4.bar(x4 - w4,   var_aucs,  w4, color='#44cc88', alpha=0.85, label='Variance')
ax4.bar(x4,        clip_aucs, w4, color='#ff4444', alpha=0.85, label='CLIP')
ax4.bar(x4 + w4,   comb_aucs, w4, color='#ffaa00', alpha=0.85, label='Combined')
ax4.axhline(0.75, color='#ffcc44', lw=1, ls='--', alpha=0.7)
ax4.axhline(0.50, color='#888888', lw=0.8, ls=':', alpha=0.5)
ax4.set_xticks(x4)
ax4.set_xticklabels([c.replace('-',' ').replace('Concept','Conc.') for c in cat_labels],
                     color='white', fontsize=8)
ax4.set_ylim(0.3, 1.05)
ax4.set_ylabel('ROC-AUC', color='white', fontsize=10)
ax4.set_title('Per-Category AUC (External)\nVariance strong across all testable categories',
              color='white', fontsize=9, fontweight='bold')
ax4.legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')

cats_all = ["Single-Concept","Compositional","Negation","Ambiguous","Conflicting"]
cat_clip_means = []
cat_clip_stds  = []
for cat in cats_all:
    vals = [r["clip_score"] for r in phase8_results if r["category"]==cat]
    cat_clip_means.append(np.mean(vals))
    cat_clip_stds.append(np.std(vals))

colors_cat = ['#4488ff','#44cc88','#ffaa00','#ff4444','#aa44ff']
bars5 = ax5.bar(range(len(cats_all)), cat_clip_means, yerr=cat_clip_stds,
                color=colors_cat, alpha=0.85, capsize=4,
                error_kw={'color':'white','elinewidth':1.5})
ax5.axhline(np.mean([r["clip_score"] for r in phase8_results if r["hallucinated"]]),
            color='#ff4444', lw=2, ls='--', label=f'Hall. mean CLIP')
ax5.axhline(np.mean([r["clip_score"] for r in phase8_results if not r["hallucinated"]]),
            color='#44cc88', lw=2, ls='--', label=f'Clean mean CLIP')

ax5.set_xticks(range(len(cats_all)))
ax5.set_xticklabels([c.replace('-',' ').replace('Concept','Conc.')
                      .replace('Compositional','Comp.')
                      for c in cats_all], color='white', fontsize=8)
ax5.set_ylabel('Mean CLIP Score', color='white', fontsize=10)
ax5.set_title('CLIP Score by Category (External)\nHigh overlap between hall/clean CLIP scores',
              color='white', fontsize=9, fontweight='bold')
ax5.legend(fontsize=8, facecolor='#2a2a2a', labelcolor='white')

ax6.axis('off')
table_data = [
    ["Metric",                    "Phase 3",      "Phase 8"],
    ["Dataset size",              "7,000",        "10,000"],
    ["Hallucination rate",        "8.2%",         "0.97%"],
    ["Variance AUC",              "0.8766",       "0.9585 ↑"],
    ["CLIP AUC",                  "0.8725",       "0.4751 ↓"],
    ["Combined AUC",              "0.8979",       "0.9604 ↑"],
    ["5-fold CV",                 "0.8962±0.048", "0.9377±0.032"],
    ["Cohen's d (variance)",      "0.82",         "1.879 ↑"],
    ["Mann-Whitney p",            "2.70e-197",    "6.34e-55"],
    ["Variance mean",             "0.002732",     "0.002754"],
    ["CLIP mean",                 "0.2401",       "0.2406"],
    ["Generalised?",              "—",            "✓ YES"],
]

table = ax6.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center', loc='center',
    cellColours=[['#1a2a1a','#1a1a3a','#1a3a1a']]*11,
    colColours=['#1a3a6a','#1a3a6a','#1a3a6a']
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1, 1.45)
for (row, col), cell in table.get_celld().items():
    cell.set_text_props(color='white')
    cell.set_edgecolor('#444444')
    if row > 0 and col == 2:
        text = cell.get_text().get_text()
        if '↑' in text:   cell.get_text().set_color('#44cc88')
        elif '↓' in text: cell.get_text().set_color('#ff4444')
        elif '✓' in text: cell.get_text().set_color('#44ff88')

ax6.set_title('Phase 8 vs Phase 3: Key Metrics',
              color='white', fontsize=10, fontweight='bold', pad=12)

fig.suptitle(
    'Phase 8 — External Validation: Trajectory Variance Generalises to Unseen Prompts\n'
    'Variance AUC: 0.877 → 0.959 (↑)   |   '
    'CLIP AUC: 0.873 → 0.475 (↓ on diverse prompts)   |   '
    'Combined AUC: 0.898 → 0.960 (↑)',
    color='white', fontsize=12, fontweight='bold', y=0.98
)

os.makedirs("Phase8_Results/figures", exist_ok=True)
plt.savefig("Phase8_Results/figures/figure9_generalisation.png",
            dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.close()
print("Figure 9 saved → Phase8_Results/figures/figure9_generalisation.png")

Figure 9 saved → Phase8_Results/figures/figure9_generalisation.png


In [14]:
print("=" * 60)
print("PHASE 8 COMPLETION CHECKLIST")
print("=" * 60)

checks = [
    ("10,000 images generated",
     len([f for f in os.listdir(IMG_DIR) if f.endswith(".png")]) == 10000),
    ("10,000 trajectories saved",
     len([f for f in os.listdir(TRAJ_DIR) if f.endswith(".npy")]) == 10000),
    ("phase8_metadata.json saved",
     os.path.exists("Phase8_Data/phase8_metadata.json")),
    ("clip_scores.json saved",
     os.path.exists("Phase8_Results/clip_scores.json")),
    ("yolo_results.json saved",
     os.path.exists("Phase8_Results/yolo_results.json")),
    ("phase8_labels.json saved",
     os.path.exists("Phase8_Results/phase8_labels.json")),
    ("phase8_summary.json saved",
     os.path.exists("Phase8_Results/phase8_summary.json")),
    ("Figure 9 saved",
     os.path.exists("Phase8_Results/figures/figure9_generalisation.png")),
]

all_pass = True
for label, result in checks:
    status = "✓" if result else "✗ FAIL"
    if not result: all_pass = False
    print(f"  [{status}] {label}")

print()
print("KEY FINDINGS:")
print(f"  • Variance AUC:  0.8766 → {auc_var_p8:.4f}  ({'↑ IMPROVED' if auc_var_p8 > auc_var_p3 else '↓ dropped'})")
print(f"  • Combined AUC:  0.8979 → {auc_comb_p8:.4f}  ({'↑ IMPROVED' if auc_comb_p8 > auc_comb_p3 else '↓ dropped'})")
print(f"  • CLIP AUC:      0.8725 → {auc_clip_p8:.4f}  (↓ fails on diverse external prompts)")
print(f"  • Cohen's d:     0.82   → {d_p8:.4f}  (↑ stronger separation)")
print(f"  • Verdict: Trajectory variance GENERALISES. CLIP does not.")
print()
if all_pass:
    print("✓ PHASE 8 COMPLETE — Ready for Phase 9")
else:
    print("✗ Some checks failed — review above")

PHASE 8 COMPLETION CHECKLIST
  [✓] 10,000 images generated
  [✓] 10,000 trajectories saved
  [✓] phase8_metadata.json saved
  [✓] clip_scores.json saved
  [✓] yolo_results.json saved
  [✓] phase8_labels.json saved
  [✓] phase8_summary.json saved
  [✓] Figure 9 saved

KEY FINDINGS:
  • Variance AUC:  0.8766 → 0.9585  (↑ IMPROVED)
  • Combined AUC:  0.8979 → 0.9604  (↑ IMPROVED)
  • CLIP AUC:      0.8725 → 0.4751  (↓ fails on diverse external prompts)
  • Cohen's d:     0.82   → 1.8791  (↑ stronger separation)
  • Verdict: Trajectory variance GENERALISES. CLIP does not.

✓ PHASE 8 COMPLETE — Ready for Phase 9
